# Разбор матча: интерактивная витрина визуализаций

Тестовый ноутбук поверх слоя **gold** платформы: выбираете лигу, сезон, матч, команду и игрока —
графики во вкладках перерисовываются. Только `pandas` + `matplotlib` + `ipywidgets`,
без футбольных библиотек. Данные читаются из Trino:

| Что | Таблица gold | Источник |
|---|---|---|
| События с координатами (SPADL, поле 0–100 × 0–100, атака слева направо) | `fct_event` | WhoScored |
| Удары с xG | `fct_shot` | Understat |
| Командная статистика по матчам | `fct_team_match` | FotMob / FBref |
| Статистика игроков по матчам | `fct_player_match` | FotMob / FBref |
| Справочники | `dim_match`, `dim_team`, `dim_player` | — |

Вкладки:
1. **Losses** — потери в своей трети и какие из них привели к удару соперника
2. **xG timeline** — кумулятивный xG матча и территориальный «моментум»
3. **Pass map** — карта передач игрока и его перцентили по лиге за сезон
4. **Shots & entries** — удары и входы в 20-метровую зону у ворот
5. **Pass network** — сети передач обеих команд
6. **League form** — изменение формы всех команд лиги: последние 7 туров против начала сезона

Запускайте все ячейки (Run → Run All Cells); панель выбора и вкладки — в конце.
Графики живут в виджетах, поэтому на GitHub они не видны — снимки есть на лендинге.

In [1]:
import os
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd
import trino
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Ellipse, Arc, FancyArrowPatch
import ipywidgets as w
from IPython.display import display

warnings.filterwarnings("ignore")
plt.ioff()  # фигуры показываем явно, внутри виджетов

# В JupyterHub платформы переменные TRINO_* уже в окружении (docs/ANALYST_ONBOARDING.md).
# TRINO_VERIFY=false — только для локального запуска с самоподписанным сертификатом.
_conn = trino.dbapi.connect(
    host=os.environ.get("TRINO_HOST", "trino"),
    port=int(os.environ.get("TRINO_PORT", "443")),
    user=os.environ.get("TRINO_USER", "analyst_svc"),
    http_scheme="https",
    auth=trino.auth.BasicAuthentication(
        os.environ.get("TRINO_USER", "analyst_svc"), os.environ["TRINO_PASSWORD"]
    ),
    verify=os.environ.get("TRINO_VERIFY", "true").lower() != "false",
    catalog="iceberg",
    schema="gold",
)


def q(sql: str) -> pd.DataFrame:
    cur = _conn.cursor()
    cur.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


def sql_list(values) -> str:
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)

## Стиль и поле

Тёмная тема и поле, нарисованное вручную в единицах WhoScored (0–100), с пропорциями 105 × 68 м.

In [2]:
BG, PANEL, LINE, TEXT, MUTED = "#0f1218", "#161b24", "#6b7386", "#e8eaf0", "#8d95a8"
OK, BAD, GOLD = "#2dd4bf", "#f87171", "#fbbf24"
HOME_C, AWAY_C = "#f5f5f5", "#7dd3fc"  # хозяева / гости

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": PANEL, "savefig.facecolor": BG,
    "axes.edgecolor": LINE, "axes.labelcolor": TEXT, "text.color": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": "#2a3140",
    "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False,
    "figure.dpi": 110,
})

PITCH_L, PITCH_W = 105.0, 68.0  # метры
KX, KY = 100 / PITCH_L, 100 / PITCH_W


def dist_to_goal_m(x, y):
    xm, ym = np.asarray(x) / KX, np.asarray(y) / KY
    return np.hypot(PITCH_L - xm, PITCH_W / 2 - ym)


def draw_pitch(ax, lc=LINE, lw=1.0):
    ax.add_patch(Rectangle((0, 0), 100, 100, fill=False, ec=lc, lw=lw))
    ax.plot([50, 50], [0, 100], color=lc, lw=lw)
    for x0, sign in ((0, 1), (100, -1)):
        ax.add_patch(Rectangle((x0 if sign > 0 else x0 - 16.5 * KX, 50 - 20.16 * KY),
                               16.5 * KX, 40.32 * KY, fill=False, ec=lc, lw=lw))
        ax.add_patch(Rectangle((x0 if sign > 0 else x0 - 5.5 * KX, 50 - 9.16 * KY),
                               5.5 * KX, 18.32 * KY, fill=False, ec=lc, lw=lw))
        ax.add_patch(Rectangle((x0 - 2 if sign > 0 else x0, 50 - 3.66 * KY),
                               2, 7.32 * KY, fill=False, ec=lc, lw=lw))  # ворота
        px = x0 + sign * 11 * KX
        ax.plot(px, 50, ".", color=lc, ms=3)
        ax.add_patch(Arc((px, 50), 2 * 9.15 * KX, 2 * 9.15 * KY,
                         theta1=-53 if sign > 0 else 127, theta2=53 if sign > 0 else 233,
                         color=lc, lw=lw))
    ax.add_patch(Ellipse((50, 50), 2 * 9.15 * KX, 2 * 9.15 * KY, fill=False, ec=lc, lw=lw))
    ax.plot(50, 50, ".", color=lc, ms=3)
    ax.set_xlim(-3, 103)
    ax.set_ylim(-3, 103)
    ax.set_aspect(PITCH_W / PITCH_L)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    return ax


def arrow(ax, x0, y0, x1, y1, color, lw=1.2, alpha=0.9, ms=8):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=ms,
                                 color=color, lw=lw, alpha=alpha, shrinkA=0, shrinkB=0))


def caption(fig, text):
    fig.text(0.01, 0.005, text, color=MUTED, fontsize=8, ha="left", va="bottom")

## Загрузка данных

Всё кэшируется по ключу (лига/сезон/матч), чтобы смена выбора не ходила в Trino повторно.

In [3]:
coverage = q('''
SELECT league, season, count(DISTINCT match_id) AS matches
FROM fct_event WHERE league NOT LIKE 'INT-%' GROUP BY 1, 2 ORDER BY 1, 2
''')
LEAGUES = sorted(coverage.league.unique())


def seasons_of(league):
    return sorted(coverage[coverage.league == league].season, reverse=True)


@lru_cache(maxsize=32)
def matches_of(league, season):
    df = q(f'''
    SELECT m.match_id, m.match_date, h.team_name AS home_name, a.team_name AS away_name,
           m.home_score, m.away_score
    FROM dim_match m
    JOIN dim_team h ON h.team_id = m.home_team_id
    JOIN dim_team a ON a.team_id = m.away_team_id
    WHERE m.league = '{league}' AND m.season = '{season}' AND m.is_completed
      AND m.match_id IN (SELECT DISTINCT match_id FROM fct_event
                         WHERE league = '{league}' AND season = '{season}')
    ORDER BY m.match_date DESC, h.team_name
    ''')
    df["label"] = [f"{r.match_date:%d.%m.%Y}  {r.home_name} {r.home_score}–{r.away_score} {r.away_name}"
                   for r in df.itertuples()]
    return df


@lru_cache(maxsize=16)
def load_match(match_id):
    meta = q(f'''
    SELECT m.match_id, m.match_date, m.league, m.season, m.gameweek,
           m.home_team_id, h.team_name AS home_name, m.home_score,
           m.away_team_id, a.team_name AS away_name, m.away_score
    FROM dim_match m
    JOIN dim_team h ON h.team_id = m.home_team_id
    JOIN dim_team a ON a.team_id = m.away_team_id
    WHERE m.match_id = '{match_id}'
    ''').iloc[0]
    events = q(f'''
    SELECT event_id, team_id, player_id, period, minute, x, y, end_x, end_y,
           action, _action_source_note AS note, outcome_success AS success
    FROM fct_event
    WHERE match_id = '{match_id}' AND period IN ('FirstHalf', 'SecondHalf')
    ORDER BY event_id
    ''').reset_index(drop=True)
    events["seq"] = np.arange(len(events))
    events["player_id"] = events.player_id.fillna("")
    shots = q(f'''
    SELECT s.team_id, s.player_id, p.player_name, s.minute, s.x * 100 AS x, s.y * 100 AS y,
           s.xg, s.situation, s.result, s.is_goal
    FROM fct_shot s LEFT JOIN dim_player p ON p.player_id = s.player_id
    WHERE s.match_id = '{match_id}'
    ORDER BY s.minute
    ''')
    shots = shots.reindex(columns=["team_id", "player_id", "player_name", "minute", "x", "y",
                                   "xg", "situation", "result", "is_goal"]) \
        .astype({"x": float, "y": float, "xg": float, "minute": float})
    shots["is_goal"] = shots.is_goal.fillna(False).astype(bool)
    pm = q(f'''
    SELECT player_id, team_id, minutes_played, passes FROM fct_player_match
    WHERE match_id = '{match_id}'
    ''').reindex(columns=["player_id", "team_id", "minutes_played", "passes"])
    pm = pm.dropna(subset=["player_id"])
    ids = sorted(set(events.player_id[events.player_id.ne("")]) | set(pm.player_id))
    names = q(f"SELECT player_id, player_name FROM dim_player WHERE player_id IN ({sql_list(ids)})") \
        if ids else pd.DataFrame(columns=["player_id", "player_name"])
    # список игроков: по событиям (есть всегда), минуты — из fct_player_match, если есть
    players = (events[events.player_id.ne("")].groupby(["player_id", "team_id"])
               .agg(actions=("seq", "size"), passes=("action", lambda a: (a == "pass").sum()))
               .reset_index()
               .merge(pm[["player_id", "minutes_played"]], on="player_id", how="left")
               .merge(names, on="player_id", how="left"))
    players["player_name"] = players.player_name.fillna(players.player_id)
    return {
        "meta": meta, "events": events, "shots": shots, "players": players,
        "home": meta.home_team_id, "away": meta.away_team_id,
        "names": {meta.home_team_id: meta.home_name, meta.away_team_id: meta.away_name},
        "colors": {meta.home_team_id: HOME_C, meta.away_team_id: AWAY_C},
        "title": f"{meta.home_name} {meta.home_score}–{meta.away_score} {meta.away_name} · "
                 f"{meta.league.split('-', 1)[1]} · {meta.match_date:%d.%m.%Y}",
    }


@lru_cache(maxsize=8)
def season_team_matches(league, season):
    return q(f'''
    WITH sx AS (
      SELECT match_id, team_id,
             sum(CASE WHEN situation = 'open_play' THEN xg ELSE 0 END) AS op_xg
      FROM fct_shot WHERE league = '{league}' AND season = '{season}' GROUP BY 1, 2
    )
    SELECT t.team_id, d.short_name, t.date, t.gameweek, t.opponent_id,
           coalesce(f.op_xg, 0) AS op_xg_for, coalesce(a.op_xg, 0) AS op_xg_against
    FROM fct_team_match t
    JOIN dim_team d ON d.team_id = t.team_id
    LEFT JOIN sx f ON f.match_id = t.match_id AND f.team_id = t.team_id
    LEFT JOIN sx a ON a.match_id = t.match_id AND a.team_id = t.opponent_id
    WHERE t.league = '{league}' AND t.season = '{season}' AND t.is_completed
    ORDER BY t.team_id, t.date
    ''')


@lru_cache(maxsize=8)
def season_player_metrics(league, season, min_minutes=900):
    sp = q(f'''
    SELECT player_id, sum(minutes_played) AS minutes,
           sum(passes) AS passes, sum(passes_completed) AS passes_completed,
           sum(key_passes) AS key_passes, sum(xa) AS xa, sum(xg_chain) AS xg_chain,
           sum(xg_buildup) AS xg_buildup, sum(ball_recoveries) AS ball_recoveries,
           sum(tackles) + sum(interceptions) AS tackles_int, sum(dribbles_won) AS dribbles_won
    FROM fct_player_match
    WHERE league = '{league}' AND season = '{season}'
    GROUP BY 1 HAVING sum(minutes_played) >= {min_minutes}
    ''')
    if sp.empty:
        return None, None

    def per90(col):
        return sp[col] / sp.minutes * 90

    metrics = pd.DataFrame({
        "Passes /90": per90("passes"),
        "Pass accuracy %": sp.passes_completed / sp.passes * 100,
        "Key passes /90": per90("key_passes"),
        "xA /90": per90("xa"),
        "xG chain /90": per90("xg_chain"),
        "xG buildup /90": per90("xg_buildup"),
        "Ball recoveries /90": per90("ball_recoveries"),
        "Tackles + int. /90": per90("tackles_int"),
        "Dribbles won /90": per90("dribbles_won"),
    })
    metrics.index = sp.player_id
    return metrics, metrics.rank(pct=True) * 100

## Графики

Каждая функция получает загруженный матч `M` и возвращает фигуру. Определения:
потеря — неудачная передача/кросс/ввод, неудачный обыгрыш или `bad_touch` на своих 40 % поля;
«привела к удару» — соперник бил в следующих 10 действиях; вход в зону — передача/кросс,
завершившийся в 20 м от центра ворот; территория — доля действий команды в финальной трети
за 5-минутный отрезок.

In [4]:
LOSS_PASS = {"pass", "cross", "throw_in", "goalkick", "freekick_short", "freekick_crossed",
             "corner_short", "corner_crossed"}
SHOT_ACTIONS = ["shot", "shot_freekick"]


def plot_losses(M):
    ev = M["events"]
    is_loss = (
        ev.action.eq("bad_touch")
        | (ev.action.isin(LOSS_PASS) & ~ev.success.fillna(True))
        | (ev.action.eq("take_on") & ~ev.success.fillna(True))
    )
    losses = ev[is_loss & (ev.x <= 40)].copy()

    def led_to_shot(row, window=10):
        nxt = ev.iloc[row.seq + 1 : row.seq + 1 + window]
        nxt = nxt[nxt.period == row.period]
        return bool(((nxt.team_id != row.team_id) & nxt.action.isin(SHOT_ACTIONS)).any())

    losses["led_to_shot"] = losses.apply(led_to_shot, axis=1) if len(losses) else False
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    for ax, team in zip(axes, (M["home"], M["away"])):
        draw_pitch(ax)
        ax.add_patch(Rectangle((0, 0), 40, 100, fc=BAD, alpha=0.06, ec="none"))
        d = losses[losses.team_id == team]
        plain, star = d[~d.led_to_shot], d[d.led_to_shot]
        ax.scatter(plain.x, plain.y, s=45, c=M["colors"][team], alpha=0.75, ec=BG, lw=0.5, label="loss")
        ax.scatter(star.x, star.y, s=260, marker="*", c=BAD, ec=BG, lw=0.5,
                   label="loss → opponent shot ≤10 actions")
        ax.set_title(f"{M['names'][team]}: {len(d)} losses in own third, {len(star)} led to a shot")
        ax.annotate("own goal", (0.5, 50), xytext=(3, 92), color=MUTED, fontsize=8)
        ax.legend(loc="lower right", fontsize=8)
    fig.suptitle(M["title"], x=0.01, ha="left", fontsize=13, fontweight="bold")
    caption(fig, "Losses: unsuccessful pass/cross/throw-in/take-on, bad touch, dispossessed, error · own 40 % of the pitch · gold.fct_event (WhoScored)")
    fig.tight_layout()
    return fig


def plot_timeline(M, BIN=5):
    ev, shots = M["events"], M["shots"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1.3], "hspace": 0.08})
    max_min = int(max(ev.minute.max(), shots.minute.max() if len(shots) else 0)) + 1
    for team in (M["home"], M["away"]):
        s = shots[shots.team_id == team].sort_values("minute")
        minutes = np.concatenate([[0], s.minute.values, [max_min]])
        cum = np.concatenate([[0], s.xg.cumsum().values, [s.xg.sum()]])
        ax1.step(minutes, cum, where="post", color=M["colors"][team], lw=2.2,
                 label=f"{M['names'][team]}  {s.xg.sum():.2f} xG")
        g = s[s.is_goal]
        for i, (_, r) in enumerate(g.iterrows()):
            yv = s.loc[: r.name, "xg"].sum()
            ax1.plot(r.minute, yv, "o", ms=9, mfc=GOLD, mec=BG)
            late = r.minute > max_min - 15
            ax1.annotate(f"{r.player_name} {int(r.minute)}'", (r.minute, yv),
                         xytext=(-8 if late else 6, -14 if i % 2 else 8), ha="right" if late else "left",
                         textcoords="offset points", fontsize=8, color=GOLD)
    ax1.set_ylim(bottom=0)
    if shots.empty:
        ax1.set_ylim(0, 1)
        ax1.text(0.5, 0.5, "no Understat shots for this match", transform=ax1.transAxes,
                 ha="center", color=MUTED)
    ax1.axvline(45, color=LINE, lw=0.8, ls="--")
    ax1.set_ylabel("cumulative xG")
    ax1.legend(loc="upper left")
    ax1.set_title(M["title"])
    ax1.grid(True, axis="y", lw=0.5, alpha=0.5)

    e = ev[ev.action.ne("unknown") & ev.player_id.ne("")].copy()
    e["bin"] = (e.minute // BIN) * BIN
    ft = e[e.x >= 200 / 3].groupby(["bin", "team_id"]).size().unstack(fill_value=0)
    ft = ft.reindex(range(0, max_min + BIN, BIN), fill_value=0)
    h, a = ft.get(M["home"], 0), ft.get(M["away"], 0)
    tilt = ((h - a) / (h + a).replace(0, np.nan)).fillna(0)
    colors = [M["colors"][M["home"]] if v >= 0 else M["colors"][M["away"]] for v in tilt]
    ax2.bar(tilt.index + BIN / 2, tilt, width=BIN * 0.9, color=colors, alpha=0.85)
    ax2.axhline(0, color=LINE, lw=0.8)
    ax2.axvline(45, color=LINE, lw=0.8, ls="--")
    ax2.set_ylim(-1, 1)
    ax2.set_ylabel("territory")
    ax2.set_xlabel("minute")
    ax2.set_xlim(0, max_min)
    ax2.text(0.01, 0.9, f"↑ {M['names'][M['home']]}", transform=ax2.transAxes, fontsize=8, color=M["colors"][M["home"]])
    ax2.text(0.01, 0.05, f"↓ {M['names'][M['away']]}", transform=ax2.transAxes, fontsize=8, color=M["colors"][M["away"]])
    caption(fig, f"xG: gold.fct_shot (Understat) · territory: share of each team's actions in the attacking third per {BIN}-minute bin, gold.fct_event (WhoScored)")
    return fig


def plot_pass_map(M, player_id):
    ev = M["events"]
    p = M["players"].set_index("player_id").loc[player_id]
    passes = ev[(ev.player_id == player_id) & ev.action.isin(["pass", "cross"])]
    metrics, pct = season_player_metrics(M["meta"].league, M["meta"].season)

    fig = plt.figure(figsize=(14, 6.2))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 1], wspace=0.3)
    ax = fig.add_subplot(gs[0])
    draw_pitch(ax)
    for ok, color in ((True, OK), (False, BAD)):
        d = passes[passes.success.fillna(False) == ok]
        for _, r in d.iterrows():
            arrow(ax, r.x, r.y, r.end_x, r.end_y, color, lw=1.1, alpha=0.8 if ok else 0.9)
    n_ok = int(passes.success.fillna(False).sum())
    mins = f" · {int(p.minutes_played)} min" if pd.notna(p.minutes_played) else ""
    ax.set_title(f"{p.player_name} ({M['names'][p.team_id]})\n{len(passes)} passes, "
                 f"{n_ok} completed ({n_ok / max(len(passes), 1):.0%}){mins}", fontsize=11)
    ax.plot([], [], color=OK, label="completed")
    ax.plot([], [], color=BAD, label="incomplete")
    ax.legend(loc="lower right", fontsize=8)
    ax.annotate("attacking →", (97, 96), ha="right", color=MUTED, fontsize=8)

    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor(BG)
    for s in ax2.spines.values():
        s.set_visible(False)
    if metrics is None or player_id not in metrics.index:
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.text(0.5, 0.5, "no season percentiles\n(fct_player_match: <900 min or no data)",
                 transform=ax2.transAxes, ha="center", color=MUTED)
    else:
        labels = list(metrics.columns)[::-1]
        for i, m in enumerate(labels):
            pc, v = pct.loc[player_id, m], metrics.loc[player_id, m]
            ax2.plot([0, 100], [i, i], color="#2a3140", lw=6, solid_capstyle="round")
            ax2.plot([0, pc], [i, i], color=OK if pc >= 50 else BAD, lw=6, solid_capstyle="round", alpha=0.9)
            ax2.plot(pc, i, "o", ms=9, mfc=TEXT, mec=BG)
            ax2.text(103, i, f"{v:.2f}  ·  {pc:.0f}th", va="center", fontsize=8.5, color=MUTED)
        ax2.set_yticks(range(len(labels)))
        ax2.set_yticklabels(labels, fontsize=9)
        ax2.set_xlim(-2, 135)
        ax2.set_ylim(-0.7, len(labels) - 0.3)
        ax2.set_xticks([0, 25, 50, 75, 100])
        ax2.tick_params(axis="x", labelsize=8)
        ax2.axvline(50, color=LINE, lw=0.6, ls=":")
        ax2.set_title(f"{M['meta'].league.split('-', 1)[1]} {M['meta'].season} percentiles · "
                      f"{len(metrics)} players ≥ 900 min")
    caption(fig, "Pass map: gold.fct_event (WhoScored) · percentiles: gold.fct_player_match, per 90 minutes")
    return fig


def plot_shots_entries(M, DEEP_M=20):
    ev, shots = M["events"], M["shots"]
    entries = ev[ev.action.isin(["pass", "cross"]) & ev.end_x.notna()].copy()
    entries = entries[(dist_to_goal_m(entries.end_x, entries.end_y) <= DEEP_M)
                      & (dist_to_goal_m(entries.x, entries.y) > DEEP_M)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    for ax, team in zip(axes, (M["home"], M["away"])):
        draw_pitch(ax)
        ax.add_patch(Arc((100, 50), 2 * DEEP_M * KX, 2 * DEEP_M * KY, theta1=90, theta2=270,
                         color=GOLD, lw=1, ls="--", alpha=0.6))
        e = entries[entries.team_id == team]
        for _, r in e.iterrows():
            arrow(ax, r.x, r.y, r.end_x, r.end_y, OK if r.success else BAD, lw=1.0, alpha=0.75, ms=7)
        s = shots[shots.team_id == team]
        ax.scatter(s.x, s.y, s=40 + s.xg * 900, c=M["colors"][team], alpha=0.55, ec=BG, lw=0.6,
                   label="shot (size = xG)")
        g = s[s.is_goal]
        ax.scatter(g.x, g.y, s=60 + g.xg * 900, marker="*", c=GOLD, ec=BG, lw=0.6, label="goal", zorder=5)
        n_ok = int(e.success.fillna(False).sum())
        ax.set_title(f"{M['names'][team]}: {len(s)} shots, {s.xg.sum():.2f} xG · "
                     f"{len(e)} deep entries ({n_ok} completed)")
        ax.legend(loc="lower left", fontsize=8)
        ax.set_xlim(45, 103)
    fig.suptitle(M["title"], x=0.01, ha="left", fontsize=13, fontweight="bold")
    caption(fig, f"Shots: gold.fct_shot (Understat) · deep entries: passes/crosses ending within {DEEP_M} m of goal centre, gold.fct_event (WhoScored)")
    fig.tight_layout()
    return fig


def plot_network(M, min_actions=25, min_edge=3):
    ev, players = M["events"], M["players"]

    def one(ax, team):
        ev_t = ev[(ev.team_id == team) & ev.player_id.ne("")]
        pl = players[players.team_id == team]
        core = set(pl[pl.minutes_played >= 45].player_id) if pl.minutes_played.notna().any() \
            else set(pl[pl.actions >= min_actions].player_id)
        pos = ev_t[ev_t.player_id.isin(core)].groupby("player_id")[["x", "y"]].mean()
        names = pl.set_index("player_id").player_name
        nxt = ev.shift(-1)
        ok = ev_t[ev_t.action.eq("pass") & ev_t.success.fillna(False)].index
        edges = pd.DataFrame({"src": ev.loc[ok, "player_id"], "dst": nxt.loc[ok, "player_id"],
                              "same": nxt.loc[ok, "team_id"] == team})
        edges = edges[edges.same & edges.dst.ne("") & edges.src.isin(core) & edges.dst.isin(core)]
        edges = edges[edges.src != edges.dst].groupby(["src", "dst"]).size()
        edges = edges.groupby(lambda k: tuple(sorted(k))).sum() if len(edges) else edges
        n_pass = ev_t[ev_t.action.eq("pass")].groupby("player_id").size().reindex(pos.index, fill_value=0)
        draw_pitch(ax)
        for (a, b), n in edges[edges >= min_edge].items():
            ax.plot([pos.x[a], pos.x[b]], [pos.y[a], pos.y[b]], color=M["colors"][team],
                    lw=0.5 + n * 0.35, alpha=0.35, solid_capstyle="round", zorder=1)
        ax.scatter(pos.x, pos.y, s=80 + n_pass * 6, c=M["colors"][team], ec=BG, lw=1, zorder=3)
        for pid, r in pos.iterrows():
            ax.annotate(str(names.get(pid, pid)).split()[-1], (r.x, r.y), xytext=(0, 9),
                        textcoords="offset points", ha="center", fontsize=7.5, color=TEXT, zorder=4)
        ax.set_title(f"{M['names'][team]} · pass network ({int(n_pass.sum())} passes)")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    for ax, team in zip(axes, (M["home"], M["away"])):
        one(ax, team)
    fig.suptitle(M["title"], x=0.01, ha="left", fontsize=13, fontweight="bold")
    caption(fig, "gold.fct_event (WhoScored): node = mean position of a player's actions, edge = completed passes between the pair (≥ 3)")
    fig.tight_layout()
    return fig


def plot_form(league, season, LAST_N=7):
    tm = season_team_matches(league, season).copy()
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.grid(True, lw=0.5, alpha=0.6)
    if tm.empty or tm.op_xg_for.sum() == 0:
        ax.text(0.5, 0.5, "no team/shot data for this season", transform=ax.transAxes,
                ha="center", color=MUTED)
        return fig
    tm["is_recent"] = tm.groupby("team_id").cumcount(ascending=False) < LAST_N
    form = (tm.groupby(["team_id", "short_name", "is_recent"])[["op_xg_for", "op_xg_against"]]
            .mean().unstack("is_recent")).dropna()
    for (team, name), row in form.iterrows():
        x0, y0 = row[("op_xg_for", False)], row[("op_xg_against", False)]
        x1, y1 = row[("op_xg_for", True)], row[("op_xg_against", True)]
        c = OK if (x1 - y1) > (x0 - y0) else BAD
        arrow(ax, x0, y0, x1, y1, c, lw=1.6, ms=14, alpha=0.9)
        ax.plot(x0, y0, "o", color=c, ms=3, alpha=0.6)
        ax.annotate(name, (x1, y1), xytext=(5, 4), textcoords="offset points",
                    fontsize=8.5, color=TEXT, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xlabel("open-play xG per game (for)")
    ax.set_ylabel("open-play xG per game (against) — inverted")
    ax.set_title(f"{league.split('-', 1)[1]} {season} · form change: last {LAST_N} games (head) vs. games before (tail)")
    ax.text(0.99, 0.02, "green: xG difference improved · red: worsened",
            transform=ax.transAxes, ha="right", color=MUTED, fontsize=8)
    caption(fig, "gold.fct_team_match + gold.fct_shot (Understat, situation = open_play)")
    fig.tight_layout()
    return fig

## Панель выбора

Лига → сезон → матч → команда → игрок. Смена матча перерисовывает все вкладки,
смена игрока — только «Pass map», смена сезона — ещё и «League form».

In [5]:
DEFAULT = {"league": "ITA-Serie A", "season": "2526", "match": "83708428"}

league_dd = w.Dropdown(options=LEAGUES, value=DEFAULT["league"] if DEFAULT["league"] in LEAGUES else LEAGUES[0],
                       description="Лига", layout=w.Layout(width="230px"))
season_dd = w.Dropdown(description="Сезон", layout=w.Layout(width="150px"))
match_dd = w.Dropdown(description="Матч", layout=w.Layout(width="520px"))
team_dd = w.Dropdown(description="Команда", layout=w.Layout(width="230px"))
player_dd = w.Dropdown(description="Игрок", layout=w.Layout(width="300px"))
status = w.HTML()

TABS = ["Losses", "xG timeline", "Pass map", "Shots & entries", "Pass network", "League form"]
outs = {t: w.Output() for t in TABS}
tabs = w.Tab(children=[outs[t] for t in TABS])
for i, t in enumerate(TABS):
    tabs.set_title(i, t)

_state = {"updating": False}


def show(name, make):
    out = outs[name]
    with out:
        out.clear_output(wait=True)
        try:
            fig = make()
            display(fig)
            plt.close(fig)
        except Exception as exc:  # чтобы одна вкладка не ломала остальные
            print(f"{name}: {type(exc).__name__}: {exc}")


def render_match():
    M = load_match(match_dd.value)
    status.value = f"<b>{M['title']}</b> · events: {len(M['events'])}, shots: {len(M['shots'])}"
    show("Losses", lambda: plot_losses(M))
    show("xG timeline", lambda: plot_timeline(M))
    show("Shots & entries", lambda: plot_shots_entries(M))
    show("Pass network", lambda: plot_network(M))
    render_player()


def render_player():
    if player_dd.value:
        show("Pass map", lambda: plot_pass_map(load_match(match_dd.value), player_dd.value))


def render_form():
    show("League form", lambda: plot_form(league_dd.value, season_dd.value))


def set_options(dd, options, value=None):
    dd.options = options
    values = [v for _, v in options]
    if values:
        dd.value = value if value in values else values[0]


def on_league(_=None):
    _state["updating"] = True
    seasons = seasons_of(league_dd.value)
    set_options(season_dd, [(s, s) for s in seasons], DEFAULT["season"])
    _state["updating"] = False
    on_season()


def on_season(_=None):
    _state["updating"] = True
    df = matches_of(league_dd.value, season_dd.value)
    set_options(match_dd, list(zip(df.label, df.match_id)), DEFAULT["match"])
    _state["updating"] = False
    render_form()
    on_match()


def on_match(_=None):
    if _state["updating"]:
        return
    _state["updating"] = True
    M = load_match(match_dd.value)
    set_options(team_dd, [(M["names"][t], t) for t in (M["home"], M["away"])])
    fill_players(M)
    _state["updating"] = False
    render_match()


def fill_players(M):
    pl = M["players"][M["players"].team_id == team_dd.value].sort_values("passes", ascending=False)
    set_options(player_dd, [(f"{r.player_name} ({int(r.passes)} passes)", r.player_id) for r in pl.itertuples()])


def on_team(_=None):
    if _state["updating"]:
        return
    _state["updating"] = True
    fill_players(load_match(match_dd.value))
    _state["updating"] = False
    render_player()


def on_player(_=None):
    if not _state["updating"]:
        render_player()


league_dd.observe(on_league, names="value")
season_dd.observe(on_season, names="value")
match_dd.observe(on_match, names="value")
team_dd.observe(on_team, names="value")
player_dd.observe(on_player, names="value")

display(w.VBox([w.HBox([league_dd, season_dd, match_dd]), w.HBox([team_dd, player_dd]), status, tabs]))
on_league()

## Что ещё можно построить на этих же данных

- Карта прессинга и возвратов мяча (`ball_recovery`, `tackle`, `interception`) по зонам.
- Сравнение двух игроков «радаром» по перцентилям из вкладки Pass map.
- Хронология сезона: xG за/против по турам со скользящим средним (`fct_team_match`).
- Тепловая карта касаний игрока и «сонары» передач по направлениям.
- Модель угрозы (xT) поверх `fct_event` — тогда моментум станет «не-ударным xG».